## **Análisis componentes para crear `DataNode` y `NodeManager`**

### **`Dockerfile`**

**`FROM hadoop-base-image:latest`**: Esta línea indica que la imagen que se va a construir se basará en la imagen base que anteriormente creamos, **hadoop-base-image**. En Docker, una imagen base proporciona el sistema de archivos inicial y los binarios necesarios para ejecutar una aplicación. En este caso, `la imagen base es Ubuntu + Actualizacion de paquetes + Instalacion de paquetes necesarios para Hadoop + Hadoop 3.3.6 + entre otros`.

**`MAINTAINER Alfonso Perez Lino`**: Esta línea se utiliza para especificar el nombre del mantenedor o creador de la imagen. En versiones anteriores de Docker, esta línea era importante para proporcionar información sobre quién mantenía o creaba la imagen. Sin embargo, a partir de Docker 1.13, la instrucción **MAINTAINER** está obsoleta en favor de las etiquetas **LABEL**. Ahora, la información del mantenedor se suele agregar como una etiqueta de metadatos en lugar de utilizar la instrucción **MAINTAINER**.

**`LABEL creador="Alfonso Perez Lino"`**: Esta línea agrega metadatos a la imagen resultante. Los metadatos proporcionan información adicional sobre la imagen, como el nombre del creador, la versión, la descripción, etc. En este caso, se está utilizando la etiqueta creador para indicar el nombre del creador de la imagen. Esta información puede ser útil para otros usuarios que trabajen con la imagen, ya que proporciona contexto sobre su origen y su propósito.

In [ ]:
# Trabajamos sobre la imagen creada anteriormente
FROM hadoop-base-image:latest

LABEL creador="Alfonso Perez Lino"

**`USER root`**: Esta línea indica que se cambiará al usuario root dentro del entorno de la imagen Docker. El usuario root es el superusuario en sistemas basados en Unix y tiene privilegios completos para realizar cualquier acción en el sistema. Al cambiar al usuario root, se obtienen permisos y privilegios completos dentro del entorno de la imagen, lo que permite ejecutar comandos que pueden requerir permisos elevados, como instalar paquetes, configurar el sistema, y realizar otras tareas administrativas.

In [ ]:
# Cambiamos al usuario root
USER root

**`ENV HADOOP_VERSION 3.3.6`**: Esta línea define una variable de entorno llamada **HADOOP_VERSION** con el valor **3.3.6**. Las variables de entorno se utilizan para almacenar información que puede ser utilizada por scripts, aplicaciones o comandos dentro del entorno de la imagen Docker. En este caso, la variable **HADOOP_VERSION** es utilizada para especificar la versión de Hadoop que se utilizará en la imagen.

**`ENV BASE_DIR /opt/bd`**: Esta línea define otra variable de entorno llamada **BASE_DIR** con el valor **/opt/bd**. Esta variable se utiliza para especificar el directorio base en el cual se instalarán o almacenarán componentes relacionados con Big Data en la imagen Docker.

**`ENV LOG_TAG "[DNNM Hadoop_${HADOOP_VERSION}]:"`**: Aquí se define la variable **LOG_TAG** con un valor que parece ser un tag para los logs. Utiliza la variable **HADOOP_VERSION** definida anteriormente para incluir la versión de Hadoop en el tag.

**`ENV HADOOP_HOME ${BASE_DIR}/hadoop/`**: Esta línea establece la variable de entorno **HADOOP_HOME** con el valor **${BASE_DIR}/hadoop/**. **HADOOP_HOME** representa la ubicación del directorio principal de instalación de Hadoop en el contenedor Docker.
Se basa en la variable de entorno **BASE_DIR**, que permite una configuración más flexible y portátil del directorio de instalación de Hadoop.

**`ENV HADOOP_CONF_DIR ${HADOOP_HOME}/etc/hadoop/`**: Esta línea establece la variable de entorno **HADOOP_CONF_DIR** con el valor **${HADOOP_HOME}/etc/hadoop/**. **HADOOP_CONF_DIR** representa la ubicación del directorio de configuración de Hadoop en el contenedor Docker. Se basa en la variable de entorno **HADOOP_HOME**, lo que proporciona una forma consistente de definir la ruta del directorio de configuración de Hadoop.

**`ENV DATA_DIR /var/data/hadoop/hdfs`**: Esta línea establece la variable de entorno **DATA_DIR** con el valor **/var/data/hadoop/hdfs**. **DATA_DIR** representa la ubicación del directorio de datos de Hadoop en el contenedor Docker. Este directorio puede ser utilizado para almacenar datos de Hadoop, como datos de HDFS u otros datos relacionados con el funcionamiento de Hadoop.

In [ ]:
# Establecemos las variables
ENV HADOOP_VERSION 3.3.6
ENV BASE_DIR /opt/bd
ENV LOG_TAG "[DNNM Hadoop_${HADOOP_VERSION}]:"
ENV HADOOP_HOME ${BASE_DIR}/hadoop/  # Recordar que este directorio "/opt/bd/hadoop" que corresponde a un "enlace simbolico"
ENV HADOOP_CONF_DIR ${HADOOP_HOME}/etc/hadoop/
ENV DATA_DIR /var/data/hadoop/hdfs

**`RUN echo "$LOG_TAG Creando carpetas para los datos de HDFS del DataNode" && \`**: Esta línea utiliza la instrucción **RUN** para ejecutar un comando en el contenedor Docker durante la construcción de la imagen. El comando **echo** simplemente imprime un mensaje en la salida estándar para informar sobre la acción que se está realizando. En este caso, imprime un mensaje que indica que se están creando carpetas para los datos de HDFS del DataNode. La cadena **$LOG_TAG** corresponde a una variable de entorno que contiene información log adicional, pero no está definida explícitamente en las líneas proporcionadas. Puede ser una variable definida previamente en el Dockerfile o en el entorno de construcción.

**`mkdir -p ${DATA_DIR}/nn`**: Este comando **mkdir** crea un directorio llamado **nn** dentro del directorio especificado por la variable de entorno `${DATA_DIR}`. La opción **-p** indica a **mkdir** que cree los directorios padres si no existen. Esto garantiza que `${DATA_DIR}` y **nn** se creen incluso si `${DATA_DIR}` no existe previamente.

**`chown -R hdadmin:hadoop ${DATA_DIR}`**: Este comando chown cambia el propietario y el grupo del directorio `${DATA_DIR}` y todos sus archivos y subdirectorios al usuario **hdadmin** y al grupo **hadoop**. La opción **-R** indica que la operación se debe realizar de forma recursiva en todos los archivos y subdirectorios dentro de `${DATA_DIR}`. Esto es útil para garantizar que los archivos y directorios necesarios para el funcionamiento de HDFS estén propiedad del usuario y grupo adecuados. 

En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

- Imprimen un mensaje log.
- Crean un directorio para los datos del DataNode de HDFS.
- Establecen los permisos adecuados en el directorio de datos de HDFS y sus subdirectorios para que sean propiedad del usuario y grupo correctos (**hdadmin:hadoop**).

In [ ]:
# Creamos las carpetas para los datos de HDFS del DataNode y hacemos el usuario hdadmin propietario de ellas 
# En un entorno de producción se deberían crear varias carpetas en particiones separadas
RUN echo "$LOG_TAG Creando carpetas para los datos de HDFS del DataNode" && \
    mkdir -p ${DATA_DIR}/dn && chown -R hdadmin:hadoop ${DATA_DIR}

**`RUN echo "$LOG_TAG Creando la carpeta para los ficheros de log" && \`**: Esta línea utiliza la instrucción **RUN** para ejecutar un comando en el contenedor Docker durante la construcción de la imagen. El comando **echo** imprime un mensaje en la salida estándar para informar sobre la acción que se está realizando. En este caso, imprime un mensaje que indica que se está creando un directorio para los archivos log. La cadena **$LOG_TAG** se utilizará como prefijo del mensaje de registro.

**`mkdir ${HADOOP_HOME}/logs`**: Este comando **mkdir** crea un directorio llamado **logs** dentro del directorio especificado por la variable de entorno **${HADOOP_HOME}**. Esto probablemente sea el directorio donde Hadoop almacenará sus archivos log.

En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

- Imprimen un mensaje log indicando la creación del directorio de archivos log.
- Crean el directorio para los archivos log en la ubicación especificada por **HADOOP_HOME**.

In [ ]:
# Creamos la carpeta para los ficheros de log
RUN echo "$LOG_TAG Creando la carpeta para los ficheros de log" && \
    mkdir ${HADOOP_HOME}/logs

**`RUN echo "$LOG_TAG Copiando ficheros de configuración y script de inicio"`**: Esta línea utiliza la instrucción **RUN** para ejecutar un comando en el contenedor Docker durante la construcción de la imagen. El comando **echo** imprime un mensaje en la salida estándar para informar sobre la acción que se está realizando. En este caso, imprime un mensaje que indica que se están copiando archivos de configuración y script de inicio.

**`COPY Config-files/core-site.xml ${HADOOP_CONF_DIR}/core-site.xml`**: Esta línea utiliza la instrucción **COPY** para copiar el archivo **core-site.xml** desde la ubicación **Config-files/core-site.xml** en el sistema de archivos del host al directorio **${HADOOP_CONF_DIR}/core-site.xml** en el sistema de archivos del contenedor Docker. **core-site.xml** es un archivo de configuración de Hadoop que contiene configuraciones para la comunicación entre los componentes del clúster Hadoop.

**`COPY Config-files/hdfs-site-datanode.xml ${HADOOP_CONF_DIR}/hdfs-site.xml`**: Esta línea utiliza la instrucción **COPY** para copiar el archivo **hdfs-site-datanode.xml** desde la ubicación **Config-files/hdfs-site-datanode.xml** en el sistema de archivos del host al directorio **${HADOOP_CONF_DIR}/hdfs-site.xml** en el sistema de archivos del contenedor Docker. **hdfs-site-datanode.xml** es otro archivo de configuración de Hadoop que contiene configuraciones específicas del DataNode de Hadoop.

**`COPY Config-files/yarn-site-nodemanager.xml ${HADOOP_CONF_DIR}/yarn-site.xml`**: Esta línea utiliza la instrucción **COPY** para copiar el archivo **yarn-site.xml** desde la ubicación **Config-files/yarn-site-resourcemanager.xml** en el sistema de archivos del host al directorio **${HADOOP_CONF_DIR}/yarn-site.xml** en el sistema de archivos del contenedor Docker.

**`COPY Config-files/mapred-site-nodemanager.xml ${HADOOP_CONF_DIR}/mapred-site.xml`**: Esta línea utiliza la instrucción **COPY** para copiar el archivo **mapred-site.xml** desde la ubicación **Config-files/mapred-site-resourcemanager.xml** en el sistema de archivos del host al directorio **${HADOOP_CONF_DIR}/mapred-site.xml** en el sistema de archivos del contenedor Docker.

En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

- Imprimen un mensaje log indicando la copia de archivos de configuración y scripts de inicio.
- Copian archivos de configuración desde el sistema de archivos del host al sistema de archivos del contenedor Docker para configurar adecuadamente el entorno de Hadoop en el contenedor.

**`core.xml`**

El archivo **core-site.xml** es un archivo de configuración importante en el ecosistema de Apache Hadoop. Su función principal es configurar opciones generales que afectan al núcleo del sistema de archivos Hadoop (Hadoop Common). Aquí hay algunas de las configuraciones comunes que pueden encontrarse en **core-site.xml**:

1. `Configuración del sistema de archivos Hadoop (HDFS)`: Esto incluye la ubicación del namenode (el nodo maestro que gestiona el sistema de archivos distribuido) y la configuración de los bloques de datos, como el tamaño predeterminado del bloque y la cantidad de replicas.

2. `Configuración de la seguridad`: Puede incluir configuraciones para la autenticación y autorización, como la elección de mecanismos de autenticación (por ejemplo, Kerberos) y la configuración de la autorización de acceso a los recursos del sistema de archivos.

3. `Configuración de recursos de red y almacenamiento`: Puede incluir configuraciones relacionadas con la utilización de la red, como la configuración del puerto de comunicación, así como configuraciones relacionadas con el almacenamiento, como la ubicación de los directorios temporales.

4. `Configuración del framework de procesamiento de datos MapReduce:` Aunque la mayoría de las configuraciones de MapReduce se realizan en el archivo mapred-site.xml, algunas configuraciones comunes también pueden estar presentes en core-site.xml, como la configuración del framework predeterminado de MapReduce.

5. `Configuración de ubicaciones de logs y directorios temporales`: Puede incluir configuraciones que especifiquen dónde se deben almacenar los registros del sistema y los archivos temporales generados por Hadoop.

En resumen, el archivo core-site.xml es esencial para configurar aspectos fundamentales del sistema de archivos distribuido y otros componentes centrales de Apache Hadoop. Contiene configuraciones críticas que afectan el funcionamiento y el rendimiento del clúster de Hadoop en su conjunto.

**`hdfs-site-datanode.xml`**

El archivo **`hdfs-site.xml`** es un archivo de configuración específico de Apache Hadoop HDFS (Hadoop Distributed File System). Su función principal es definir las configuraciones relacionadas con el sistema de archivos distribuido HDFS en un clúster de Hadoop. Aquí hay una descripción de algunas configuraciones comunes que pueden encontrarse en `hdfs-site.xml` y su función:

1. `Configuración de la replicación de datos`: HDFS almacena los datos en múltiples réplicas para garantizar la tolerancia a fallos y la disponibilidad de datos. En hdfs-site.xml, puedes configurar el número de réplicas que se deben mantener para cada bloque de datos. Esto se realiza mediante la propiedad dfs.replication.

2. `Configuración del tamaño del bloque de datos`: HDFS divide los archivos en bloques de datos de un tamaño fijo y los distribuye en diferentes nodos del clúster. Puedes especificar el tamaño de estos bloques de datos mediante la propiedad dfs.blocksize.

3. `Configuración de la ubicación del almacenamiento de datos`: Puedes especificar la ubicación en el sistema de archivos del sistema de archivos local donde se almacenarán los datos de HDFS. Esto se hace mediante la propiedad dfs.datanode.data.dir.

4. `Configuración de los nombres de los nodos secundarios y de los puntos de montaje`: En un clúster HDFS, puedes tener nodos secundarios que ayudan en la administración del sistema de archivos. Puedes configurar la ubicación de los nombres de los nodos secundarios y los puntos de montaje de HDFS mediante las propiedades dfs.namenode.name.dir y dfs.namenode.edits.dir.

5. `Configuración de cuotas y permisos de acceso`: Puedes configurar cuotas para el espacio de almacenamiento y el número de archivos para los usuarios y grupos, así como establecer permisos de acceso para los archivos y directorios en HDFS.

6. `Configuración de auditoría y registro de eventos`: Puedes habilitar el registro de eventos de auditoría para rastrear las operaciones realizadas en el sistema de archivos HDFS y configurar la ubicación de los registros de auditoría.

En resumen, el archivo hdfs-site.xml es esencial para configurar aspectos fundamentales del sistema de archivos distribuido HDFS en un clúster de Hadoop. Contiene configuraciones críticas que afectan el rendimiento, la disponibilidad y la seguridad de los datos almacenados en HDFS.

**`yarn-site-nodemanager.xml`**

El archivo **yarn-site-resourcemanager.xml** es un archivo de configuración específico de Apache Hadoop YARN (Yet Another Resource Negotiator). YARN es el administrador de recursos en Hadoop que gestiona y asigna recursos de manera eficiente a las aplicaciones que se ejecutan en un clúster de Hadoop.

El propósito del archivo yarn-site-resourcemanager.xml es configurar los parámetros específicos del ResourceManager de YARN. El ResourceManager es el componente principal de YARN que es responsable de la asignación de recursos y la programación de tareas en el clúster.

Algunas de las configuraciones comunes que pueden encontrarse en yarn-site-resourcemanager.xml incluyen:

1. `Capacidad de los recursos del clúster`: Define cuántos recursos de CPU y memoria están disponibles en el clúster para asignar a las aplicaciones.

2. `Políticas de cola`: Permite configurar colas de aplicaciones con diferentes prioridades y límites de recursos.

3. `Seguridad`: Configuraciones relacionadas con la seguridad, como la autenticación y la autorización.

4. `Localización de nodos`: Define cómo se comunican los nodos con el ResourceManager y entre sí.

5. `Retención de registros y métricas`: Configuraciones relacionadas con la retención de registros y métricas de aplicaciones ejecutadas en el clúster.

En resumen, el archivo **yarn-site-resourcemanager.xml** es esencial para configurar y ajustar el comportamiento del ResourceManager de YARN en un clúster de Hadoop, y su función principal es definir cómo se asignan y gestionan los recursos en el clúster.

**`mapred-site-nodemanager.xml`**

El archivo **mapred-site.xml** es un archivo de configuración específico de Apache Hadoop MapReduce. Su función principal es configurar el comportamiento del framework de procesamiento de datos MapReduce en un clúster de Hadoop. Aquí hay algunas de las configuraciones comunes que se pueden encontrar en **mapred-site.xml**:

1. `Configuración del framework de ejecución MapReduce`: Esto incluye la configuración de parámetros relacionados con cómo se ejecutan los trabajos MapReduce en el clúster, como la configuración del framework de ejecución (por ejemplo, MapReduce clásico o YARN), la asignación de recursos y la administración de tareas.

2. `Configuración del rastreador de tareas (JobTracker o ResourceManager)`: MapReduce clásico utiliza un componente llamado JobTracker para administrar y programar trabajos MapReduce. Si se está utilizando YARN como administrador de recursos, entonces ResourceManager desempeña un papel similar. En mapred-site.xml, se pueden configurar opciones relacionadas con estas componentes, como la ubicación del JobTracker o del ResourceManager y la configuración de la alta disponibilidad.

3. `Configuración de tareas de MapReduce`: Puede incluir configuraciones relacionadas con la ejecución de tareas de MapReduce, como el número máximo de intentos de reintentos, la cantidad de memoria asignada a tareas individuales y la configuración de las colas de trabajos.

4. `Configuración de la localización de los datos intermedios y de salida`: Es posible especificar dónde se deben almacenar los datos intermedios y los resultados de las tareas de MapReduce dentro del sistema de archivos distribuido (HDFS).

5. `Configuración de la compresión de datos`: Puede incluir configuraciones relacionadas con la compresión de datos para reducir el tamaño de los datos intermedios y de salida generados por las tareas de MapReduce.

En resumen, el archivo mapred-site.xml es esencial para configurar el comportamiento y el rendimiento del framework de procesamiento de datos MapReduce en un clúster de Hadoop. Contiene configuraciones críticas que afectan cómo se ejecutan y administran los trabajos MapReduce en el clúster.

In [ ]:
# Copiamos los archivos de configuración y el script de inicio
RUN echo "$LOG_TAG Copiando los archivos de configuración y el script de inicio"
COPY Config-files/core-site.xml ${HADOOP_CONF_DIR}/core-site.xml
COPY Config-files/hdfs-site-datanode.xml ${HADOOP_CONF_DIR}/hdfs-site.xml
COPY Config-files/yarn-site-nodemanager.xml ${HADOOP_CONF_DIR}/yarn-site.xml
COPY Config-files/mapred-site-nodemanager.xml ${HADOOP_CONF_DIR}/mapred-site.xml

**`COPY Config-files/start-daemons-dnnm.sh ${BASE_DIR}/start-daemons.sh`**:Esta línea utiliza la instrucción **COPY** para copiar el archivo **start-daemons-dnnm.sh** desde la ubicación **Config-files/start-daemons-dnnm.sh** en el sistema de archivos del host al directorio **${BASE_DIR}/start-daemons.sh** en el sistema de archivos del contenedor Docker. **Config-files/start-daemons-dnnm.sh** es la ruta en el sistema de archivos del host donde se encuentra el archivo que se desea copiar.

**${BASE_DIR}/start-daemons.sh** es la ruta en el sistema de archivos del contenedor Docker donde se copiará el archivo. **BASE_DIR** es una variable de entorno que se espera que contenga la ruta base del directorio deseado en el contenedor. El archivo se copiará con el nombre **start-daemons.sh** en el directorio especificado.

En resumen, esta línea de comando en el Dockerfile realiza la siguiente acción:

- Copia un archivo desde el sistema de archivos del host al sistema de archivos del contenedor Docker para su uso posterior en el entorno del contenedor.

In [ ]:
# Script de inicio
COPY Config-files/start-daemons-dnnm.sh ${BASE_DIR}/start-daemons.sh

**`RUN chmod +x ${BASE_DIR}/start-daemons.sh && \`**: Esta línea utiliza la instrucción **RUN** para ejecutar dos comandos en el contenedor Docker durante la construcción de la imagen. El primer comando, `chmod +x ${BASE_DIR}/start-daemons.sh`, cambia los permisos del archivo **start-daemons.sh** ubicado en el directorio especificado por `${BASE_DIR}` para que sea ejecutable (**+x**). Esto significa que después de ejecutar este comando, el archivo **start-daemons.sh** será ejecutable en el contenedor Docker.
La notación **${BASE_DIR}** se refiere a una variable de entorno que se espera que contenga la ruta base del directorio deseado en el contenedor.

**`chown -R hdadmin:hadoop ${BASE_DIR}`**: El segundo comando establece el propietario y el grupo del directorio `${BASE_DIR}` y todos sus archivos y subdirectorios al usuario **hdadmin** y al grupo **hadoop**. La opción **-R** indica que la operación se realizará de forma recursiva en todos los archivos y subdirectorios dentro de **${BASE_DIR}**. Esto es útil para garantizar que el directorio y sus contenidos sean propiedad del usuario y grupo correctos, lo que puede ser importante para el correcto funcionamiento de ciertas aplicaciones o servicios dentro del contenedor Docker.

En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

- Cambian los permisos del archivo **start-daemons.sh** para que sea ejecutable.
- Establecen el propietario y el grupo del directorio **${BASE_DIR}** y sus contenidos al usuario **hdadmin** y al grupo **hadoop**.

In [ ]:
RUN chmod +x ${BASE_DIR}/start-daemons.sh && \
    chown -R hdadmin:hadoop ${BASE_DIR}

Se indican 4 puertos para cada servicio. Lo más seguro es que cada uno representara a cada contenedor en el caso de crear 4 DataNode-NodeManager.

In [ ]:
# Establecemos los puertos para los servicios web del Datanode
EXPOSE 9864 9865 9866 9867

# Establecemos los puertos para los servicios web del NodeManager
EXPOSE 8040 8042 8044 8048

# # Establecemos los puertos para los servicios web del MapReduce
EXPOSE 50000-50050 50100-50200

La línea de comando `USER hdadmin` en un Dockerfile de Docker se utiliza para establecer el usuario bajo el cual se ejecutarán los comandos y procesos dentro del contenedor Docker cuando se inicie. Aquí está el desglose:

**`USER hdadmin`**: Esta línea establece el usuario predeterminado que se utilizará para ejecutar los comandos y procesos dentro del contenedor Docker. **hdadmin** es el nombre de usuario que se establece como usuario predeterminado. Cuando esta línea se encuentra en un Dockerfile y se construye la imagen, el usuario predeterminado dentro del contenedor se cambiará a **hdadmin** después de que se inicien los contenedores basados en esa imagen. Esto es útil para especificar qué usuario debe ser utilizado dentro del contenedor Docker para fines de seguridad y gestión de permisos. Cambiar al usuario **hdadmin** después de iniciar el contenedor puede ser beneficioso para restringir los privilegios de los procesos dentro del contenedor, especialmente si no es necesario que se ejecuten como el usuario **root** por razones de seguridad.

Es importante tener en cuenta que el usuario **hdadmin** debe existir dentro del contenedor. De lo contrario, se producirán errores durante la construcción o ejecución del contenedor si se especifica un usuario que no existe.

En resumen, la línea de comando `USER hdadmin` en un Dockerfile establece el usuario predeterminado que se utilizará para ejecutar comandos y procesos dentro del contenedor Docker. 

In [ ]:
# Cambiamos al usuario hdadmin
USER hdadmin

**`ENV JAVA_HOME /usr/lib/jvm/java-8-openjdk-amd64/`**: Esta línea establece la variable de entorno **JAVA_HOME** con la ruta al directorio de instalación de Java en el contenedor.

**`ENV PATH ${PATH}:${HADOOP_HOME}/bin/:${HADOOP_HOME}/sbin/`**: Esta línea modifica la variable de entorno **PATH** para incluir los directorios donde residen los binarios de Hadoop (`${HADOOP_HOME}/bin/` y `${HADOOP_HOME}/sbin/`). Esto permite que los comandos de Hadoop estén disponibles en la ruta de búsqueda del sistema.

En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

- Cambian al usuario hdadmin para ejecutar comandos que no requieren permisos especiales.
- Establecen variables de entorno para configurar el entorno de desarrollo y definir rutas importantes.


**`¿A que se refieren con PATH?`**

La variable de entorno `PATH` en Linux es una cadena que contiene una lista de directorios separados por dos puntos (`:`). Esta variable es utilizada por el sistema operativo para determinar los directorios en los que debe buscar los ejecutables de los comandos cuando se invocan en la línea de comandos.

**Función de la variable PATH**

`1. Localización de Ejecutables:`Cuando un usuario ejecuta un comando en la línea de comandos sin especificar una ruta completa, el sistema operativo busca el ejecutable correspondiente en los directorios listados en la variable PATH, en el orden en que aparecen.

`2. Facilitar la Ejecución de Comandos:`Sin PATH, los usuarios tendrían que proporcionar la ruta completa de cada ejecutable para ejecutar un comando, lo que sería incómodo y poco práctico.

`3. Modularidad y Flexibilidad:` Los usuarios pueden modificar su PATH para incluir directorios personalizados, lo que les permite añadir nuevos comandos sin necesidad de mover los ejecutables a los directorios estándar del sistema.

**Ejemplo de Uso**

Supongamos que la variable **PATH** tiene el siguiente valor:

In [ ]:
/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games

Cuando se ejecuta un comando, por ejemplo, **ls**, el sistema operativo busca el ejecutable **ls** en cada uno de estos directorios en orden:

1. /usr/local/sbin/ls
2. /usr/local/bin/ls
3. /usr/sbin/ls
4. /usr/bin/ls
5. /sbin/ls
6. /bin/ls
7. /usr/games/ls

El primer ejecutable **ls** que encuentra será el que se ejecutará. Si no encuentra **ls** en ninguno de estos directorios, el sistema devolverá un error indicando que el comando no fue encontrado.

**Modificación del PATH**

Los usuarios pueden modificar la variable **PATH** en su sesión actual de la terminal usando el comando export. Por ejemplo:

In [ ]:
export PATH=$PATH:/opt/myprogram/bin

Este comando añade `/opt/myprogram/bin` al final de la lista de directorios en **PATH**. A partir de este momento, el sistema también buscará ejecutables en `/opt/myprogram/bin` cuando se ejecute un comando.

Para hacer cambios permanentes, los usuarios pueden añadir la modificación de **PATH** en su archivo de configuración de shell, como **~/.bashrc** o **~/.profile**. (Esto es para hacerlo en Linux)

In [ ]:
# Definimos las variables
ENV JAVA_HOME /usr/lib/jvm/java-8-openjdk-amd64/
#ENV HADOOP_HOME ${BASE_DIR}/hadoop/ <------- Esto se repite, el usuario "root" ya creó esta variable de entorno
#ENV HADOOP_CONF_DIR ${HADOOP_HOME}/etc/hadoop/  <------- Esto se repite, el usuario "root" ya creó esta variable de entorno
ENV PATH ${PATH}:${HADOOP_HOME}/bin/:${HADOOP_HOME}/sbin/

**`WORKDIR ${HADOOP_HOME}`**: Esta línea establece el directorio de trabajo (working directory) predeterminado dentro del contenedor Docker. ${HADOOP_HOME} es una variable de entorno que se espera que contenga la ruta al directorio principal de instalación de Hadoop dentro del contenedor. Al establecer el directorio de trabajo con **WORKDIR**, cualquier comando posterior ejecutado en el Dockerfile se ejecutará en el contexto de este directorio, a menos que se especifique una ruta completa.

**`RUN echo "$LOG_TAG Iniciando servicios"`**: Esta línea utiliza la instrucción **RUN** para ejecutar un comando durante la construcción de la imagen. El comando echo simplemente imprime un mensaje en la salida estándar. En este caso, imprime un mensaje que indica que se están iniciando los servicios. La cadena **$LOG_TAG** parece ser una variable que contiene alguna información log adicional, pero no está definida explícitamente en las líneas proporcionadas. Puede ser una variable definida previamente en el Dockerfile o en el entorno de construcción.

**`CMD ["/opt/bd/start-daemons.sh"]`**: Esta línea especifica el comando predeterminado que se ejecutará cuando se inicie un contenedor basado en esta imagen. El comando se especifica como una matriz JSON, donde el primer elemento es el comando a ejecutar (**/opt/bd/start-daemons.sh**). **/opt/bd/start-daemons.sh** parece ser la ruta al script de inicio de los demonios (**daemons**) de Hadoop dentro del contenedor Docker. Este script probablemente sea utilizado para iniciar los servicios de Hadoop necesarios dentro del contenedor. En resumen, estas líneas de comando en el Dockerfile realizan las siguientes acciones:

Establecen el directorio de trabajo predeterminado dentro del contenedor.

- Imprimen un mensaje durante la construcción de la imagen.
- Especifican el comando predeterminado que se ejecutará al iniciar un contenedor basado en esta imagen.

In [ ]:
WORKDIR ${HADOOP_HOME}

RUN echo "$LOG_TAG Iniciando servicios"
CMD ["/opt/bd/start-daemons.sh"]

### **Config-files/`core-site.xml`**

El archivo **core-site.xml** en Hadoop es un archivo de configuración clave que define propiedades fundamentales para la operación del clúster Hadoop, incluyendo la configuración del sistema de archivos distribuido de Hadoop (HDFS). Vamos a desglosar y explicar los componentes del ejemplo proporcionado.

**Estructura General**

El archivo está envuelto en una etiqueta `<configuration>`, dentro de la cual hay varias etiquetas `<property>`. Cada `<property>` define una propiedad de configuración con sus respectivas etiquetas `<name>`, `<value>`, y opcionalmente `<final>`.

**Componentes del `core-site.xml`**

**1. Configuración de fs.defaultFS**
```
<property>
  <!-- nombre del Namenode -->
  <name>fs.defaultFS</name>
  <value>hdfs://namenode:9000/</value>
  <final>true</final>
</property>
```
**`<name>fs.defaultFS</name>`**: Esta propiedad define el sistema de archivos predeterminado que utilizará Hadoop. **fs.defaultFS** es el nombre de la propiedad.

**`<value>hdfs://namenode:9000/</value>`**: El valor de esta propiedad es la URI del DataNode en el clúster Hadoop. **hdfs://namenode:9000/** indica que el sistema de archivos predeterminado es **HDFS**, y que el DataNode se encuentra en el host namenode escuchando en el puerto **9000**.

- **Ejemplo de URI en Hadoop**

  En el contexto de Hadoop, un ejemplo de URI es **hdfs://namenode:9000/**. Aquí está el desglose:

  - `hdfs`: El esquema indica que se está utilizando el sistema de archivos distribuido de Hadoop (HDFS).
  - `namenode:9000`: La autoridad que especifica el host y el puerto donde está ejecutándose el DataNode.
  - `/`: La ruta, que en este caso es la raíz del sistema de archivos HDFS.

- **Función de URI en Hadoop**

  En Hadoop, las URIs se utilizan para especificar ubicaciones de recursos de manera uniforme. Por ejemplo:

  - **Configuración de fs.defaultFS**:

    - `fs.defaultFS` es una propiedad en core-site.xml que especifica el sistema de archivos predeterminado para Hadoop.
    - `Valor típico`: hdfs://namenode:9000/. Esto indica que el sistema de archivos HDFS está disponible en el host namenode en el puerto 9000.

**`<final>true</final>`**: Esta etiqueta opcional indica que la propiedad no puede ser sobrescrita por configuraciones posteriores. Es decir, su valor es final y no puede ser modificado.

**2. Configuración de hadoop.tmp.dir**
```
<property>
  <!-- Almacenamiento temporal (debe tener suficiente espacio) -->
  <name>hadoop.tmp.dir</name>
  <value>/var/tmp/hadoop-${user.name}</value>
  <final>true</final>
</property>
```

**`<name>hadoop.tmp.dir</name>`**: Esta propiedad especifica el directorio temporal que utiliza Hadoop para almacenar datos temporales.
**hadoop.tmp.dir** es el nombre de la propiedad.

**`<value>/var/tmp/hadoop-${user.name}</value>`**: El valor de esta propiedad es la ruta al directorio temporal.
`/var/tmp/hadoop-${user.name}` utiliza una variable **${user.name}**, que se expande al nombre del usuario que está ejecutando los procesos de Hadoop. Esto permite que cada usuario tenga su propio directorio temporal separado.

**`<final>true</final>`**: Igual que en el caso anterior, esta etiqueta indica que el valor de la propiedad es final y no puede ser modificado.

**Propósito de Estas Propiedades**

`fs.defaultFS`: Define el sistema de archivos predeterminado que Hadoop utilizará para almacenar y recuperar datos. Configurar correctamente esta propiedad es crucial para la operación del clúster, ya que todos los componentes de Hadoop dependerán de esta URI para acceder a HDFS.

`hadoop.tmp.dir`: Especifica el directorio utilizado para almacenamiento temporal. Este directorio debe tener suficiente espacio para manejar datos temporales utilizados por los procesos de Hadoop, como archivos de intercambio y datos intermedios generados durante las operaciones de procesamiento de datos.

Estas configuraciones son fundamentales para garantizar que Hadoop pueda localizar y utilizar sus recursos de manera eficiente y que el almacenamiento temporal esté configurado adecuadamente para evitar problemas durante la ejecución de trabajos de procesamiento de datos.

In [ ]:
<configuration>
  <property>
    <!-- nombre del Namenode -->
    <name>fs.defaultFS</name>
    <value>hdfs://namenode:9000/</value>
    <final>true</final>
  </property>
  <property>
    <!-- Almacenamiento temporal (debe tener suficiente espacio) -->
    <name>hadoop.tmp.dir</name>
    <value>/var/tmp/hadoop-${user.name}</value>
    <final>true</final>
  </property>
</configuration>

### **Config-files/`hdfs-site-datanode.xml`**

El archivo **`hdfs-site-datanode.xml`**, al igual que el archivo **hdfs-site.xml**, es un archivo de configuración específico de Hadoop que se utiliza para configurar los nodos de datos (DataNodes) en un clúster de Hadoop. Aquí hay una explicación detallada de cada componente del ejemplo proporcionado:

- `<configuration>`: Esta etiqueta marca el inicio del archivo de configuración.

- `<property>`: Esta etiqueta envuelve cada propiedad de configuración y sus valores.

- `<final>true</final>`: Esta propiedad indica si la configuración es final, es decir, si se permite o no sobrescribir su valor en tiempo de ejecución. En este caso, se establece en true, lo que significa que esta configuración no se puede cambiar una vez que se ha establecido.

- `</property>`: Esta etiqueta marca el final de la definición de la propiedad.

- `</configuration>`: Esta etiqueta marca el final del archivo de configuración.

- `<!-- Determina en qué parte del sistema de archivos local debe almacenar sus bloques un datanode DFS. Si se trata de una lista de directorios delimitada por comas, los datos se almacenarán en todos los directorios nombrados. Por rendimiento, si los nodos tienen múltiples discos, es conveniente especificar un directorio en cada uno de los discos locales -->`: Este comentario proporciona una descripción detallada de la propiedad de configuración dfs.datanode.data.dir. Explica que esta propiedad determina en qué parte del sistema de archivos local debe almacenar sus bloques un DataNode de HDFS. También indica que si se especifica una lista de directorios delimitada por comas, los datos se almacenarán en todos los directorios nombrados. Además, sugiere que, por razones de rendimiento, es beneficioso especificar un directorio en cada uno de los discos locales si los nodos tienen múltiples discos.

- `<name>dfs.datanode.data.dir</name>`: Esta propiedad define el nombre de la configuración. En este caso, es **dfs.datanode.data.dir**, que es la propiedad utilizada para especificar la ubicación en el sistema de archivos local donde se almacenarán los bloques de datos en un DataNode de HDFS.

- `<value>file:///var/data/hadoop/hdfs/dn</value>`: Esta propiedad especifica el valor de la configuración. En este caso, se establece en file:///var/data/hadoop/hdfs/dn, que es una ruta de archivo local en el sistema de archivos del nodo de datos donde se almacenarán los bloques de datos de HDFS. Aquí, file:// indica que la ruta es un archivo local, y /var/data/hadoop/hdfs/dn es la ruta completa al directorio de almacenamiento de datos del nodo de datos.

En resumen, el archivo **hdfs-site-datanode.xml** contiene una sola configuración que define la ubicación en el sistema de archivos local donde se almacenarán los bloques de datos en un nodo de datos (DataNode) de HDFS en un clúster de Hadoop. La propiedad dfs.datanode.data.dir es crucial para garantizar que los datos se almacenen de manera eficiente y confiable en el sistema de archivos local del nodo de datos.

In [ ]:
<configuration>
  <property>
  <!-- Determina en qué parte del sistema de archivos local debe almacenar sus bloques un datanode DFS. 
      Si se trata de una lista de directorios delimitada por comas, los datos se almacenarán en todos los directorios nombrados.
      Por rendimiento, si los nodos tienen múltiples discos, es conveniente especificar un directorio en cada uno de los discos locales -->
    <name>dfs.datanode.data.dir</name>
    <value>file:///var/data/hadoop/hdfs/dn</value>
    <final>true</final>
  </property>
</configuration>

### **Config-files/`yarn-site-nodemanager.xml`**

El archivo **`yarn-site-nodemanager.xml`** contiene configuraciones específicas de YARN (Yet Another Resource Negotiator) relacionadas con el NodeManager en un clúster de Hadoop. Aquí está el desglose de cada elemento del archivo:

- `<configuration>`: Esta etiqueta marca el inicio del archivo de configuración.

- `<property>`: Esta etiqueta envuelve cada propiedad de configuración y sus valores.

- `<name>`: Especifica el nombre de la propiedad de configuración.

- `<value>`: Especifica el valor de la propiedad de configuración.

- `<final>`: Indica si la propiedad es final, es decir, si no se permite sobrescribirla en tiempo de ejecución. En este caso, todas las propiedades tienen un valor de true, lo que significa que son propiedades finales.

- `</property>`: Esta etiqueta marca el final de la definición de la propiedad.

- `</configuration>`: Esta etiqueta marca el final del archivo de configuración.

- **`<name>yarn.resourcemanager.hostname</name>`**: Esta propiedad define el nombre de host del ResourceManager de YARN.

- **`<name>yarn.nodemanager.resource.detect-hardware-capabilities</name>`**: Esta propiedad indica si se debe habilitar la autodetección de las capacidades del nodo, como la memoria y la CPU.

- **`<name>yarn.nodemanager.resource.cpu-vcores</name>`**: Esta propiedad define el número de vcores (núcleos virtuales) que se pueden asignar a los contenedores en el NodeManager. Si se establece en -1 y yarn.nodemanager.resource.detect-hardware-capabilities es true, se determina automáticamente a partir del hardware.

- **`<name>yarn.nodemanager.resource.memory-mb</name>`**: Esta propiedad define la cantidad de memoria física, en megabytes (MB), que se puede asignar a los contenedores en el NodeManager. Si se establece en -1 y yarn.nodemanager.resource.detect-hardware-capabilities es true, se calcula automáticamente.

- **`<name>yarn.nodemanager.vmem-check-enabled</name>`**: Esta propiedad indica si se aplicarán límites de memoria virtual a los contenedores en el NodeManager.

- **`<name>yarn.nodemanager.aux-services</name>`**: Esta propiedad define una lista separada por comas de los servicios auxiliares implementados por los NodeManagers. En este caso, indica a los NodeManagers que deben implementar el servicio MapReduce shuffle.

- **`<name>yarn.nodemanager.aux-services.mapreduce_shuffle.class</name>`**: Esta propiedad define la clase que implementa el servicio de MapReduce shuffling.

En resumen, el archivo yarn-site-nodemanager.xml contiene configuraciones específicas del NodeManager de YARN en un clúster de Hadoop, incluyendo configuraciones relacionadas con los recursos asignados a los contenedores, la detección de capacidades de hardware, la habilitación de servicios auxiliares y otros aspectos de la gestión de recursos en el clúster. Estas configuraciones son cruciales para optimizar el rendimiento y la utilización de recursos en un clúster de Hadoop.

In [ ]:
<configuration>
<!-- Site specific YARN configuration properties -->
      <property>
        <!-- El nombre de host del ResourceManager -->
        <name>yarn.resourcemanager.hostname</name>
        <value>resourcemanager</value>
        <final>true</final>
      </property>

  <property>
    <!-- Habilitar la autodetección de las capacidades del nodo como memoria y CPU -->
    <name>yarn.nodemanager.resource.detect-hardware-capabilities</name>
    <value>true</value>
    <final>true</final>
  </property>

  <property>
    <!-- Número de vcores que se pueden asignar a los contenedores.
         Si se establece en -1 y yarn.nodemanager.resource.detect-hardware-capabilities es true, 
         se determina automáticamente a partir del hardware (valor por defecto 8) -->
    <name>yarn.nodemanager.resource.cpu-vcores</name>
    <value>-1</value>
    <final>true</final>
  </property>

  <property>
    <!-- Cantidad de memoria física, en MB, que se puede asignar a los contenedores. 
        Si se establece a -1 y yarn.nodemanager.resource.detect-hardware-capabilities es true, 
        se calcula automáticamente (valor por defecto 8192) -->
    <name>yarn.nodemanager.resource.memory-mb</name>
    <value>-1</value>
    <final>true</final>
  </property>

  <property>
    <!-- Si se aplicarán límites de memoria virtual a los contenedores (por defecto true) -->
    <name>yarn.nodemanager.vmem-check-enabled</name>
    <value>false</value>
    <final>true</final>
  </property>


 <property>
    <!-- Una lista separada por comas de los servicios auxiliares implementados ny los NodeManagers. 
        En este caso, indica a los NodeManagers que tienen que implementar el servicio MapReduce shuffle-->
    <name>yarn.nodemanager.aux-services</name>
    <value>mapreduce_shuffle</value>
    <final>true</final>
  </property>

  <property>
    <!-- Clase que implementa el servicio de MapReduce shuffling -->
    <name>yarn.nodemanager.aux-services.mapreduce_shuffle.class</name>
    <value>org.apache.hadoop.mapred.ShuffleHandler</value>
    <final>true</final>
  </property>
</configuration>

### **Config-files/`mapred-site-nodemanager.xml`**

El archivo **mapred-site.xml** (o mapred-site-nodemanager.xml, que se refiere específicamente a la configuración del nodo manager) es uno de los archivos de configuración clave en un clúster de Hadoop. Contiene configuraciones específicas para el framework MapReduce y para el nodo gestor de recursos (ResourceManager o NodeManager).

**Estructura General**

El archivo está envuelto en una etiqueta `<configuration>`, dentro de la cual hay varias etiquetas `<property>`. Cada `<property>` define una propiedad de configuración con sus respectivas etiquetas `<name>`, `<value>`, y opcionalmente `<final>`.

**Componentes del `mapred-site-nodemanager.xml`**

**1. Configuración de `mapreduce.framework.name`**

Esta propiedad especifica el **framework de ejecución** que se utilizará para ejecutar los trabajos de MapReduce. El valor predeterminado es `yarn`, que indica que se utilizará YARN para administrar los recursos. Por ejemplo:

   ```xml
                                            <property>
                                              <name>mapreduce.framework.name</name>
                                              <value>yarn</value>
                                            </property>
   ```

**2. Configuración de `yarn.app.mapreduce.am.env`**

Esta propiedad configura las variables de entorno para el proceso ApplicationMaster (AM) de MapReduce en YARN. La variable de entorno que se establece aquí es específica para el AM de MapReduce.

   ```xml
                      <property>
                        <!-- User added environment variables for the MR App Master process.
                            In this case, it specifies the location of the libraries for the MR App Master-->
                        <name>yarn.app.mapreduce.am.env</name>
                        <value>HADOOP_MAPRED_HOME=${HADOOP_HOME}</value>
                      </property>
   ```
- `HADOOP_MAPRED_HOME=${HADOOP_HOME}`: Este valor asigna la variable de entorno HADOOP_MAPRED_HOME al valor de HADOOP_HOME. Esto significa que la ubicación de las bibliotecas para el AM de MapReduce se establecerá en la misma ubicación que la variable de entorno HADOOP_HOME.

**3. Configuración de `yarn.app.mapreduce.am.resource.cpu-vcores`**

Esta propiedad especifica el número de núcleos de CPU virtuales que el ApplicationMaster (AM) de MapReduce en YARN necesita.

   ```xml
                      <property>
                        <!--  The number of virtual CPU cores the MR AppMaster needs (default value 1) -->
                        <name>yarn.app.mapreduce.am.resource.cpu-vcores</name>
                        <value>1</value>
                        <final>true</final>
                      </property>
   ```
- `1`: Este es el valor asignado a la propiedad, que indica que el AM de MapReduce necesita un solo núcleo de CPU virtual.

**4. Configuración de `yarn.app.mapreduce.am.resource.mb`**

Esta propiedad especifica la cantidad de memoria que el ApplicationMaster (AM) de MapReduce en YARN necesita.

   ```xml
                      <property>
                        <!-- The amount of memory the MR AppMaster needs (default value 1536) -->
                        <name>yarn.app.mapreduce.am.resource.mb</name>
                        <value>1024</value>
                        <final>true</final>
                      </property>
   ```
- `1024`: Este es el valor asignado a la propiedad, que indica que el AM de MapReduce necesita 1024 MB de memoria.

**5. Configuración de `mapreduce.job.heap.memory-mb.ratio`**

Esta propiedad define el ratio de tamaño de la heap del trabajo MapReduce con respecto al tamaño del contenedor.

   ```xml
                      <property>
                        <!-- The ratio of heap-size to container-size --> 
                        <name>mapreduce.job.heap.memory-mb.ratio</name>
                        <value>0.8</value>
                        <final>true</final>
                      </property>
   ```
- `0.8`: Este es el valor asignado a la propiedad, que indica que el tamaño de la heap del trabajo MapReduce será el 80% del tamaño del contenedor.

**6. Configuración de `mapreduce.map.env`**

Esta propiedad configura las variables de entorno para los procesos de mapas en el framework de MapReduce.

   ```xml
                      <property>
                        <!-- User added environment variables for the Maps. 
                            In this case, it specifies the location of the libraries for the Maps -->
                        <name>mapreduce.map.env</name>
                        <!-- A las variables si puedo indicarle un directorio, donde cualquiera de
                             de sus subdirectorios tenga las librerias para los procesos  -->
                        <value>HADOOP_MAPRED_HOME=${HADOOP_HOME}</value>
                      </property>
   ```
- `HADOOP_MAPRED_HOME=${HADOOP_HOME}`: Este valor asigna la variable de entorno HADOOP_MAPRED_HOME al valor de HADOOP_HOME. Esto significa que la ubicación de las liberias para los procesos de mapas se establecerá en la misma ubicación que la variable de entorno HADOOP_HOME.

**7. Configuración de `mapreduce.map.cpu.vcores`**

Esta propiedad determina el número de núcleos virtuales que se solicitarán para cada tarea de map en MapReduce.

   ```xml
                      <property>
                        <!-- The number of virtual cores to request for each map task (default value 1) -->
                        <name>mapreduce.map.cpu.vcores</name>
                        <value>1</value>
                        <final>true</final>
                      </property>
   ```
- `1`: lo que indica que se solicitará un núcleo virtual por tarea de map.

**8. Configuración de `mapreduce.map.java.opts`**

Se utiliza en la configuración de MapReduce en Hadoop para establecer las opciones de Java específicas para los JVM (Java Virtual Machines) que ejecutan las tareas de map.

Esta propiedad permite configurar diversas opciones relacionadas con la ejecución de las tareas de map en MapReduce. Por ejemplo, puedes especificar el tamaño máximo de la memoria heap (**-Xmx**), el tamaño inicial de la memoria heap (**-Xms**), opciones de ajuste de rendimiento, configuraciones de seguridad, etc.

   ```xml
                      <property>
                        <!-- Java options for the Maps JVMs -->
                        <name>mapreduce.map.java.opts</name>
                        <value>-Xmx1024M</value> <!-- Specify the maximum size, in bytes, of the Java heap memory -->
                        <final>true</final>
                      </property>
   ```
- `-Xmx1024M`: Esta opción de Java (-Xmx) se utiliza para especificar el tamaño máximo de la memoria heap que puede ser utilizada por el JVM. Aquí, se configura para que cada tarea de map tenga un límite de 1024 megabytes (1 gigabyte) de memoria heap.
- `true`: significa que esta configuración no puede ser modificada durante la ejecución del trabajo de MapReduce.

**9. Configuración de `mapreduce.map.memory.mb`**

Se refiere a la cantidad de memoria asignada para cada tarea de map en MapReduce en Hadoop, medida en megabytes (MB). Esta configuración es crucial para garantizar un rendimiento óptimo y una utilización eficiente de los recursos del clúster.

   ```xml
                      <property>
                        <!-- The amount of memory to request for each map task.
                            If this is not specified or is non-positive, it is calculated as 
                            (heapSize / mapreduce.heap.memory-mb.ratio) -->
                        <name>mapreduce.map.memory.mb</name>
                        <value>-1</value>
                        <final>true</final>
                      </property>
   ```
- `-1`: Esto indica que la cantidad de memoria solicitada para cada tarea de map no está especificada o es no positiva. El comentario dentro del elemento property explica que si el valor de esta propiedad no está especificado o es no positivo, se calculará en función de la propiedad mapreduce.heap.memory-mb.ratio. Esta última propiedad generalmente se utiliza para especificar la relación entre el tamaño de la memoria heap y la cantidad total de memoria asignada para MapReduce.

En resumen, este fragmento de configuración permite que la cantidad de memoria asignada a cada tarea de map sea determinada automáticamente por el sistema, basada en el cálculo descrito, si no se especifica explícitamente o si el valor especificado es no positivo.

**10. Configuración de `mapreduce.reduce.env`**

Esta propiedad se utiliza para establecer variables de entorno específicas para los procesos de reducción (reduce) en MapReduce.

   ```xml
                      <property>
                        <!-- User added environment variables for the Reduces.
                            In this case, it specifies the location of the libraries for the Reduces -->
                        <name>mapreduce.reduce.env</name>
                        <value>HADOOP_MAPRED_HOME=${HADOOP_HOME}</value>
                      </property>
   ```
- `HADOOP_MAPRED_HOME=${HADOOP_HOME}`: Es el valor asignado a la propiedad. Aquí, se establece que la variable de entorno HADOOP_MAPRED_HOME será igual al valor de la variable de entorno HADOOP_HOME. Esto significa que se están definiendo variables de entorno adicionales para los procesos de reducción, y en este caso particular, se está configurando la variable HADOOP_MAPRED_HOME para que apunte al mismo directorio que la variable HADOOP_HOME.

Establecer variables de entorno de esta manera puede ser útil para garantizar que los procesos de reducción tengan acceso a las mismas librerías u otros recursos que los procesos de map o para mantener la coherencia en la configuración del entorno de ejecución. En este caso específico, se indica que las librerías para los procesos de reducción se encuentran en el mismo directorio que las librerías principales de Hadoop (HADOOP_HOME).

**11. Configuración de `mapreduce.reduce.cpu.vcores`**

Esta propiedad se utiliza para especificar el número de núcleos virtuales que se solicitarán para cada tarea de reducción (reduce) en MapReduce.

   ```xml
                        <property>
                          <!-- The number of virtual cores to request for each reduce task (default value 1) -->
                          <name>mapreduce.reduce.cpu.vcores</name>
                          <value>1</value>
                          <final>true</final>
                        </property>
   ```
- `1`: Esto indica que se solicitará un núcleo virtual por tarea de reducción.

**12. Configuración de `mapreduce.reduce.java.opts`**

Se utiliza en la configuración de MapReduce en Hadoop para establecer las opciones de Java específicas para los JVM (Java Virtual Machines) que ejecutan las tareas de reducción (reduce).

Esta propiedad permite configurar diversas opciones relacionadas con la ejecución de las tareas de reduce en MapReduce. Por ejemplo, puedes especificar el tamaño máximo de la memoria heap (**-Xmx**), el tamaño inicial de la memoria heap (**-Xms**), opciones de ajuste de rendimiento, configuraciones de seguridad, etc.

   ```xml
                    <property>
                      <!-- Java options for the Reduces JVMs -->
                      <name>mapreduce.reduce.java.opts</name>
                      <value>-Xmx1024M</value> <!-- Xmx max size of Java Heap -->
                      <final>true</final>
                    </property>
   ```
- `-Xmx1024M`: Esta opción de Java (-Xmx) se utiliza para especificar el tamaño máximo de la memoria heap que puede ser utilizada por el JVM. En este caso, se configura para que cada JVM de tarea de reducción tenga un límite de 1024 megabytes (1 gigabyte) de memoria heap.

**13. Configuración de `mapreduce.reduce.memory.mb`**

Determina la cantidad de memoria asignada para cada tarea de reducción en megabytes (MB).

   ```xml
                  <property>
                    <!-- The amount of memory to request for each map task.
                        If this is not specified or is non-positive, it is calculated as 
                        (heapSize / mapreduce.heap.memory-mb.ratio -->
                    <name>mapreduce.reduce.memory.mb</name>
                    <value>-1</value>
                    <final>true</final>
                  </property>
                </configuration>
   ```
- `-1`: Esto indica que la cantidad de memoria asignada para cada tarea de reducción no está especificada o no es positiva. El comentario dentro del elemento property explica que si el valor de esta propiedad no está especificado o es no positivo, se calculará en función de otra propiedad llamada mapreduce.heap.memory-mb.ratio.

Esta configuración permite que el sistema calcule automáticamente la cantidad de memoria a asignar para cada tarea de reducción, lo que puede ser útil para optimizar el uso de recursos en el clúster y garantizar un rendimiento adecuado en función de la carga de trabajo y las características del entorno.

### **Config-files/`start-daemons-dnnm.sh`**

Esta línea es el shebang que indica al sistema operativo que use el intérprete de Bash para ejecutar el script:

In [ ]:
#!/bin/bash

Define la variable **HADOOP_HOME**, que apunta al directorio de instalación de Hadoop:

In [ ]:
HADOOP_HOME=/opt/bd/hadoop/

Define las variables **SERVICE1**, **SERVICE2**, **DAEMON1** y **DAEMON2**. **SERVICE1** apunta al binario del comando hdfs de Hadoop, **SERVICE2** apunta al binario del comando yarn de Hadoop y **DAEMON1** y **DAEMON2** especifican los nombres de los demonios a gestionar, en este caso, **datanode** y **nodemanager** respectivamente.

In [ ]:
SERVICE1=${HADOOP_HOME}/bin/hdfs
DAEMON1=datanode

SERVICE2=${HADOOP_HOME}/bin/yarn
DAEMON2=nodemanager

Inicia el demonio del DataNode. La variable **status** captura el código de retorno del comando (**$?**), que es **0** si el comando se ejecuta correctamente. Si **status** no es **0**, imprime un mensaje de error y termina el script con el código de error correspondiente.

**Desglose Línea por Línea**

**1. Iniciar el Servicio**:

`${SERVICE1} --daemon start ${DAEMON1}`

- `${SERVICE1}`: Esta variable contiene la ruta al binario de hdfs (el comando HDFS de Hadoop), que en este caso es ${HADOOP_HOME}/bin/hdfs.
- `--daemon start ${DAEMON1}`: Esta parte del comando indica que se debe iniciar un demonio (daemon) y el demonio a iniciar es el namenode (almacenado en la variable ${DAEMON}).

En conjunto, esta línea inicia el DataNode como un proceso en segundo plano (daemon).

**2. Capturar el Código de Retorno**:

`status=$?`

- `$?`: Esta es una variable especial en bash que contiene el código de retorno del último comando ejecutado.
- `status=$?`: Aquí se asigna el valor del código de retorno del comando anterior (es decir, el intento de iniciar el DataNode) a la variable status.

    **Código de Retorno**:

- Un código de retorno de 0 típicamente indica que el comando se ejecutó con éxito.
- Un código diferente de 0 indica que hubo algún error al ejecutar el comando.

**3. Verificar el Estado y Manejar Errores**:

```
if [ $status -ne 0 ]; then
  echo "No pudo inicializar el servicio ${DAEMON1}: $status"
  exit $status
fi
```
-`if [ $status -ne 0 ]; then`: Esta línea verifica si el valor de status es diferente de 0 (**-ne** significa "no es igual" en bash).
    
  - Si status no es 0, significa que el comando para iniciar el DataNode falló.

-`echo "No pudo inicializar el servicio ${DAEMON1}: $status"`: Si el comando falló, esta línea imprime un mensaje de error indicando que no se pudo inicializar el servicio datanode y muestra el código de error.

- `exit $status`: Termina la ejecución del script y devuelve el mismo código de error (status) que generó el fallo al intentar iniciar el DataNode.

-`fi`: Finaliza el bloque if.


**Resumen del Comportamiento**

**1. Intenta Iniciar el DataNode**:

- `hdfs --daemon start datanode` se ejecuta para iniciar el DataNode como un demonio.

**2. Captura del Código de Retorno**:

- El código de retorno del intento de iniciar el DataNode se captura en la variable status.

**3. Verificación del Éxito o Fallo**:

- Si `status` es diferente de 0, se imprime un mensaje de error y el script termina con el mismo código de error.
- Si `status` es 0, el script continúa su ejecución, ya que el DataNode se inició correctamente.

In [ ]:
# Iniciamos el demonio del DataNode y chequeamos si ha arrancado
${SERVICE1} --daemon start ${DAEMON1}
status=$?
if [ $status -ne 0 ]; then
  echo "No pudo inicializar el servicio ${DAEMON1}: $status"
  exit $status
fi

Luego, para el demonio del **NodeManager** es lo mismo:

In [ ]:
# Iniciamos el demonio del NodeManager y chequeamos si ha arrancado
${SERVICE2} --daemon start ${DAEMON2}
status=$?
if [ $status -ne 0 ]; then
  echo "No pudo inicializar el servicio ${DAEMON2}: $status"
  exit $status
fi

---

Mantiene el contenedor activo mientras el demonio datanode esté en funcionamiento:

- **while true**: Bucle infinito.
- **sleep 10**: Espera 10 segundos en cada iteración del bucle.
- **if ! ps aux | grep ${DAEMON1} | grep -q -v grep**: Verifica si el demonio **datanode** está en ejecución. Si no lo está (! invierte el resultado), imprime un mensaje de error y termina el script con un código de error **1**.
- **if ! ps aux | grep ${DAEMON2} | grep -q -v grep**: Verifica si el demonio **nodemanager** está en ejecución. Si no lo está (! invierte el resultado), imprime un mensaje de error y termina el script con un código de error **1**.

In [ ]:
# Mientras ambos demonio estén vivos, el contenedor sigue activo
while true
do 
  sleep 10
  if ! ps aux | grep ${DAEMON1} | grep -q -v grep
  then
      echo "El demonio ${DAEMON1}  ha fallado"
      exit 1
  fi
  if ! ps aux | grep ${DAEMON2} | grep -q -v grep
  then
      echo "El demonio ${DAEMON2}  ha fallado"
      exit 1
  fi
done

**`Resumen`**

Este script automatiza la configuración y ejecución del DataNode-NodeManager de Hadoop dentro de un entorno de contenedor. Realiza las siguientes acciones:

1. Define las variables de entorno necesarias.
2. Inicia el demonio del DataNode-NodeManager.
3. Verifica si el demonio se ha iniciado correctamente.
4. Mantiene el contenedor activo y monitorea el demonio del DataNode, terminando el contenedor si el demonio falla.